# Day 3.4 — Managed Memory with Mem0 (Optional)

## Before you begin

**Optional guided exposure.** Nothing here is assessed and nothing here is required. The local store you built in Day 3.3 is the complete, required path. If you do run the hosted calls, use fictional identities and synthetic content only, and put `MEM0_API_KEY` in `.env` - never in this notebook.

### Learning outcomes

- Recognise what a managed memory product does for you, and what it still leaves you to do.
- Read a captured Mem0 response and map its fields onto the local store from Day 3.3.

Architecture reference: [Day 3 diagrams D10](../diagrams/source/day_03.md).

### Expected observation

This notebook runs to completion with no key and no `mem0ai` package installed: it prints what it skipped and shows a captured example response instead.

## Concept briefing

## Managed memory products

A hosted memory product such as Mem0 extracts candidate facts from a conversation and
stores them for you. It removes plumbing - schema, extraction, retrieval, an inspection
interface - and that is a real saving.

It does not remove the duties. Consent, per-user isolation, retention, deletion and the
decision about what is worth remembering stay with the application. A product also adds a
network hop, a quota, a bill and a second copy of the data outside your control, so the
comparison is convenience against transparency and portability, not good against bad.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/safe_task_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

## Step 1 — Is the optional package available?

`mem0ai` is not part of the course requirements, so this cell expects it to be missing and says
so politely instead of raising.

In [ ]:
try:
    from mem0 import MemoryClient
    MEM0_INSTALLED = True
except ImportError:
    MemoryClient = None
    MEM0_INSTALLED = False
    print("Optional: pip install mem0ai to run this part live.")

MEM0_KEY = bool(os.getenv("MEM0_API_KEY"))
print("mem0ai installed :", MEM0_INSTALLED)
print("MEM0_API_KEY set :", MEM0_KEY)
print("Will call the hosted service:", MEM0_INSTALLED and MEM0_KEY)

## Step 2 — The hosted call (skipped unless both checks passed)

Mem0 takes raw conversation messages and extracts memories itself; you do not write the
extraction rules. Check the current Mem0 documentation if the SDK surface has moved on.

In [ ]:
live_result = None
if MEM0_INSTALLED and MEM0_KEY:
    try:
        client = MemoryClient(api_key=os.environ["MEM0_API_KEY"])
        messages = [{"role": "user",
                     "content": "For this fictional lab, I prefer meetings after 10:00."}]
        live_result = client.add(messages, user_id="course_fictional_asha")
        print("Add   :", live_result)
        print("Search:", client.search("When should meetings be scheduled?",
                                       filters={"user_id": "course_fictional_asha"}))
    except Exception as exc:
        print("Hosted call failed, continuing with the captured example:", exc)
else:
    print("Hosted call skipped. Day 3.3's local store is the fallback and the required path.")

## Step 3 — A captured example response

So that everyone sees what the platform returns, here is a response captured from a course demo
run, trimmed and re-typed as a Python literal. Field names can change between SDK versions -
treat this as the *shape*, not a contract.

In [ ]:
import json                      # used to pretty-print the captured dictionaries

captured_add_response = {
    "results": [
        {"id": "0d7c9f2e-...-a41b",
         "memory": "Prefers meetings after 10:00",
         "event": "ADD"}
    ]
}

captured_search_response = {
    "results": [
        {"id": "0d7c9f2e-...-a41b",
         "memory": "Prefers meetings after 10:00",
         "user_id": "course_fictional_asha",
         "score": 0.42,
         "created_at": "2025-01-14T09:12:44.101Z"}
    ]
}

shown = live_result if live_result is not None else captured_add_response
print("source          :", "your live call" if live_result is not None else "captured example")
print("add() returned  :", json.dumps(shown, indent=2, default=str)
      if isinstance(shown, (dict, list)) else shown)
print("\nsearch() returns:", json.dumps(captured_search_response, indent=2))
print("\nNotice what the service did for you: it turned a sentence of chat into the")
print("short third-person fact 'Prefers meetings after 10:00'. In Day 3.3 you wrote")
print("that sentence yourself when you called store.add(...).")

## Step 4 — Compare the two routes honestly

Same job, different trade-offs. Read the table row by row.

In [ ]:
rows = [
    ("who extracts the fact", "you, in store.add(...)", "the service, from raw messages"),
    ("where data lives",      "your SQLite file",       "the vendor's cloud"),
    ("inspect / delete",      "SQL you can read",       "SDK calls plus a web dashboard"),
    ("cost",                  "none",                   "quota and a bill"),
    ("latency",               "microseconds",           "a network round trip"),
    ("portability",           "a file you own",         "an export you must request"),
    ("consent and isolation", "your responsibility",    "still your responsibility"),
]
print(f"{'aspect':<22}{'local SQLite store':<26}{'Mem0 Platform'}")
print("-" * 78)
for aspect, local, hosted in rows:
    print(f"{aspect:<22}{local:<26}{hosted}")

## Step 5 — Clean-up is part of the exercise

If you did run the hosted calls, delete the synthetic record afterwards. The line is left as a
string on purpose so that nothing is deleted by accident when the cell runs in mock mode.

In [ ]:
cleanup = 'client.delete_all(user_id="course_fictional_asha")'
if MEM0_INSTALLED and MEM0_KEY:
    print("Run this yourself once you have finished inspecting the dashboard:")
    print("   ", cleanup)
else:
    print("Nothing was stored, so there is nothing to delete.")
    print("If you do run the hosted lab later, finish with:", cleanup)

### Checkpoint

**1. Mem0 wrote the memory text for you. Which responsibilities did that remove?**

<details><summary>Show answer</summary>

Only the plumbing: schema, extraction, storage and a retrieval API. Consent, per-user isolation, retention, deletion on request, and deciding what is even worth remembering all stay in your application.

</details>

**2. This notebook printed 'Hosted call skipped'. Did you miss required material?**

<details><summary>Show answer</summary>

No. The hosted lab is optional guided exposure. Everything Day 3 assesses is in Day 3.3's local store, and the captured example above shows exactly what the platform would have returned.

</details>

### Recap

- **Limitation we saw:** A managed product hides where the data went and who can read it.
- **Layer we added:** A side-by-side comparison of a transparent local store with a hosted service, using synthetic identities only.
- **Evidence it worked:** The notebook completed with no key and no package installed, printing the captured response shape and a row-by-row trade-off table.